# Ensemble: Combinación de Modelos A, B, C y D

Este notebook carga los 4 modelos entrenados de forma individual y evalúa
múltiples estrategias de ensemble:

1. **Hard Voting** -- voto mayoritario sobre las etiquetas predichas
2. **Soft Voting** -- promedio de probabilidades predichas
3. **Weighted Voting** -- voto ponderado según el score de validación cruzada
4. **Stacking** -- meta-modelo entrenado sobre predicciones de los modelos base

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_dataset, get_train_test_split
from src.evaluation import (
    evaluate_model,
    plot_full_evaluation,
    compare_models,
)
from src.model_registry import load_all_models, load_model_metadata

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Cargar Datos y Modelos

In [ ]:
df = load_dataset()
X_train, X_test, y_train, y_test = get_train_test_split(df)

In [ ]:
models = load_all_models()
print(f"Models loaded: {list(models.keys())}")

## 2. Rendimiento de Modelos Individuales

Evaluar cada modelo en el conjunto de prueba para tener una línea base de comparación.

In [ ]:
individual_metrics = []
for name, model in models.items():
    m = plot_full_evaluation(model, X_test, y_test, model_name=name)
    individual_metrics.append(m)

df_individual = compare_models(individual_metrics)
df_individual

## 3. Ensemble por Hard Voting

In [ ]:
estimators = [(name, model) for name, model in models.items()]

hard_voting = VotingClassifier(
    estimators=estimators,
    voting="hard",
)

hard_voting.fit(X_train, y_train)
hard_metrics = plot_full_evaluation(
    hard_voting,
    X_test,
    y_test,
    model_name="Hard Voting",
)

## 4. Ensemble por Soft Voting

In [ ]:
soft_voting = VotingClassifier(
    estimators=estimators,
    voting="soft",
)

soft_voting.fit(X_train, y_train)
soft_metrics = plot_full_evaluation(
    soft_voting,
    X_test,
    y_test,
    model_name="Soft Voting",
)

## 5. Ensemble por Weighted Voting

Weights are derived from each model's `cv_score` only when optimization metrics are comparable.

In [ ]:
# Load CV scores and verify they are comparable.
raw_scores = []
metrics_used = []

for name in models.keys():
    meta = load_model_metadata(name)
    cv_score = float(meta.get("cv_score", 0.0))
    optimization_metric = meta.get("optimization_metric", "unknown")

    raw_scores.append(max(cv_score, 0.0))
    metrics_used.append(str(optimization_metric))
    print(f"{name}: CV score = {cv_score:.4f} ({optimization_metric})")

unique_metrics = set(metrics_used)
if len(unique_metrics) == 1 and raw_scores and sum(raw_scores) > 0:
    total = float(sum(raw_scores))
    weights = [score / total for score in raw_scores]
    print("Using normalized CV-score weights (comparable metrics).")
else:
    # If metrics differ (or scores are invalid), use equal weights to avoid biased weighting.
    weights = [1.0 / len(raw_scores)] * len(raw_scores)
    print("Using equal weights (non-comparable or missing CV-score metadata).")

print(f"Weights: {weights}")

In [ ]:
weighted_voting = VotingClassifier(
    estimators=estimators,
    voting="soft",
    weights=weights,
)

weighted_voting.fit(X_train, y_train)
weighted_metrics = plot_full_evaluation(
    weighted_voting,
    X_test,
    y_test,
    model_name="Weighted Voting",
)

## 6. Ensemble por Stacking

Usa una Regresión Logística como meta-modelo sobre las predicciones de los modelos base.

In [ ]:
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=13),
    cv=5,
    passthrough=False,
)

stacking.fit(X_train, y_train)
stacking_metrics = plot_full_evaluation(
    stacking,
    X_test,
    y_test,
    model_name="Stacking",
)

## 7. Comparación: Todos los Modelos y Ensembles

In [ ]:
all_metrics = individual_metrics + [
    hard_metrics,
    soft_metrics,
    weighted_metrics,
    stacking_metrics,
]
df_comparison = compare_models(all_metrics)
df_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_comparison.plot.barh(x="model", y="f1", ax=ax, color="steelblue")
ax.set_xlabel("F1 Score")
ax.set_title("Model Comparison: F1 Score")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Best ensemble strategy: select the top model in `df_comparison` prioritizing `f1` and checking `recall` for stroke.
- Contribution analysis: compare each ensemble against `df_individual` to verify whether the combination outperforms the best standalone model.
- Final recommendation: choose the model with the best trade-off between stroke detection (recall) and false positives (precision), using ROC-AUC as secondary support.